# EDA 003.04 — Clustering with K-Means

**Kaggle Feature Engineering Course — Lesson 4**
**Reference:** [Clustering With K-Means](https://www.kaggle.com/code/ryanholbrook/clustering-with-k-means) by Ryan Holbrook

---

## Learning Objectives

By the end of this notebook you will understand:

1. How K-Means clustering works (the algorithm, step by step)
2. Using cluster **labels as a new feature** for supervised learning
3. Using **cluster distances** as a richer set of features
4. How to choose K using the **Elbow method** and **Silhouette score**
5. The **assumptions and limitations** of K-Means

---

## The Core Idea

> In feature engineering, we use K-Means not just for clustering analysis — but as a way to **create new features** that capture the overall pattern of similarity across all input variables at once.

Two derived features:

| Feature type | Description | Use case |
|---|---|---|
| **Cluster label** | Integer 0..K-1 indicating group membership | Encodes segment; can be one-hot encoded |
| **Distance to each centroid** | K continuous features | Richer signal — captures *how far* from each prototype |

---

## How K-Means Works

```
1. Choose K centroids (random initialisation)
2. Assign each point to the nearest centroid  ← E step
3. Move each centroid to the mean of its assigned points  ← M step
4. Repeat steps 2–3 until centroids stop moving (convergence)
```

K-Means minimises **within-cluster sum of squares (WCSS)** — the total squared distance between each point and its assigned centroid.

> **Important:** Always **scale features** before K-Means. Since it uses Euclidean distance, features on larger scales will dominate.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs

sns.set_theme(style="whitegrid")
%matplotlib inline

---
## Demo 1 — K-Means Step by Step (Visualised)

In [ ]:
X_blobs, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
df = pd.DataFrame(X_blobs, columns=['x1', 'x2'])

# Fit K-Means with K=4
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(df[['x1', 'x2']])
centroids = km.cluster_centers_

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']
for i in range(4):
    mask = df['cluster'] == i
    ax.scatter(df.loc[mask, 'x1'], df.loc[mask, 'x2'], s=30, alpha=0.7,
               color=colors[i], label=f'Cluster {i}')

ax.scatter(centroids[:, 0], centroids[:, 1], s=200, c='black',
           marker='X', zorder=5, label='Centroids')
ax.set_title('K-Means Clustering (K=4)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

---
## Demo 2 — Choosing K: Elbow Method & Silhouette Score

- **Elbow method:** Plot WCSS vs K — look for the 'elbow' where adding more clusters gives diminishing returns
- **Silhouette score:** Measures how well each point fits its cluster (range -1 to +1; higher is better)

In [ ]:
wcss = []
sil  = []
K_range = range(2, 10)

for k in K_range:
    km_k = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km_k.fit_predict(X_blobs)
    wcss.append(km_k.inertia_)
    sil.append(silhouette_score(X_blobs, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(K_range, wcss, 'bo-')
axes[0].axvline(4, color='red', linestyle='--', label='True K=4')
axes[0].set_xlabel('K')
axes[0].set_ylabel('WCSS (Inertia)')
axes[0].set_title('Elbow Method')
axes[0].legend()

axes[1].plot(K_range, sil, 'go-')
axes[1].axvline(4, color='red', linestyle='--', label='True K=4')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (higher = better)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Demo 3 — Cluster Features for Supervised Learning

We simulate a house price dataset where clusters represent **neighbourhood archetypes** (cheap suburban, mid-range, luxury). Adding cluster features improves a simple model.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

np.random.seed(0)
n = 600
# 3 neighbourhood archetypes
archetype = np.random.choice([0, 1, 2], n, p=[0.4, 0.4, 0.2])
archetype_params = {
    0: (60,  10, 200000),   # small, old,  cheap
    1: (120, 20, 450000),   # medium, mid, medium
    2: (250, 5,  1200000),  # large, new,  luxury
}

area  = np.array([np.random.normal(archetype_params[a][0], 15)     for a in archetype])
age   = np.array([np.random.normal(archetype_params[a][1], 5)      for a in archetype])
price = np.array([np.random.normal(archetype_params[a][2], 50000)  for a in archetype])

df = pd.DataFrame({'area': area, 'age': age, 'price': price})

# Scale before clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['area', 'age']])

km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_label'] = km3.fit_predict(X_scaled)

# Distance to each centroid
distances = km3.transform(X_scaled)
for i in range(3):
    df[f'dist_centroid_{i}'] = distances[:, i]

target = df['price']

def cv_r2(features):
    scores = cross_val_score(LinearRegression(), df[features], target,
                             cv=5, scoring='r2')
    return scores.mean()

r2_base     = cv_r2(['area', 'age'])
r2_label    = cv_r2(['area', 'age', 'cluster_label'])
r2_dist     = cv_r2(['area', 'age'] + [f'dist_centroid_{i}' for i in range(3)])

print(f"CV R² — base features only     : {r2_base:.4f}")
print(f"CV R² — + cluster label         : {r2_label:.4f}")
print(f"CV R² — + centroid distances    : {r2_dist:.4f}")

---
## Limitations of K-Means

| Limitation | Detail |
|---|---|
| Assumes spherical clusters | Struggles with elongated or non-convex shapes |
| Sensitive to outliers | One outlier can pull a centroid significantly |
| Requires K upfront | Must decide number of clusters before fitting |
| Sensitive to scale | **Always standardise features first** |
| Non-deterministic | Results vary with different random seeds (use `n_init` > 1) |

**Alternatives:** DBSCAN (arbitrary shapes, detects outliers), Gaussian Mixture Models (soft assignments), Hierarchical Clustering.

---

## Key Takeaways

- K-Means is a **feature engineering tool**, not just a clustering analysis technique
- **Cluster labels** encode segment membership; **centroid distances** give a richer, continuous signal
- Always **scale features** before fitting K-Means
- Use **elbow + silhouette** together to pick K
- Fit on **training data only**, then transform both train and test

---

## Further Reading

- [Kaggle Tutorial — Clustering With K-Means](https://www.kaggle.com/code/ryanholbrook/clustering-with-k-means)
- [sklearn docs — KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [StatQuest — K-means clustering](https://www.youtube.com/watch?v=4b5d3muPQmA) (YouTube)
- [Understanding K-Means Clustering](https://towardsdatascience.com/understanding-k-means-clustering-in-machine-learning-6a6e67336aa1) — Towards Data Science